# Retrieval-Augmented Generation (RAG) for Conversational Recommendations

## 1. Introduction to the Target Problem

### The Challenge: Hallucination in Recommender Systems
Traditional Sequence-to-Sequence (Seq2Seq) models (like BART or T5) are excellent at generating fluent text but suffer from **hallucination** when asked for specific facts. If asked to "Recommend a sci-fi book," a standard model might invent a convincing-sounding title that doesn't exist. Furthermore, their knowledge is "frozen" in their weights; they cannot recommend new products added to a catalog after training.

### The Solution: Retrieval-Augmented Generation
This project implements the architecture proposed in **"Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" (Lewis et al., 2020)**. By decoupling the "memory" (a searchable document index) from the "processing" (the neural generator), RAG addresses two key problems:
1.  **Grounding:** The model is forced to look at real retrieved documents (book titles, prices) before generating an answer.
2.  **Updatability:** We can update the recommendation catalog simply by re-indexing the documents, without re-training the neural network.

In this notebook, we adapt RAG for a **Conversational Product Recommender**. The system retrieves real book descriptions from a dataset based on user queries and synthesizes a natural language recommendation.

## 2. Architecture Overview

### System Components
We implement a **RAG-Token** architecture, where the generator integrates retrieved context at the token level.

* **Query Encoder ($q(x)$):** A **DPR (Dense Passage Retrieval)** Question Encoder (BERT-based) that maps user queries into a dense vector space.
* **Document Index ($d(z)$):** A **FAISS** index containing pre-computed embeddings of book descriptions. We use the **DPR Context Encoder** to create these embeddings.
* **Retriever:** A mechanism that performs Maximum Inner Product Search (MIPS) to find the Top-$K$ relevant books for a given query.
* **Generator ($p(y|x, z)$):** A **BART-Large** model. It takes the query and the retrieved passages as input and generates the response token-by-token, marginalizing the probability over all retrieved documents.

### Diagram
```text
[User Query] 
      │
      ▼
[DPR Question Encoder] ───► [Vector Representation]
                                      │
                                      ▼
                            [FAISS Index Search (MIPS)]
                                      │
                             ┌────────┴────────┐
                             ▼                 ▼
                     [Retrieved Doc A]  [Retrieved Doc B] ...
                             │                 │
                             └────────┬────────┘
                                      ▼
                            [BART Generator]
                                      │
                                      ▼
                           "I recommend [Doc A Title]..."

## Implementation constraints
* Due to hardware constraints (GTX 1650 Ti, 4GB VRAM), this implementation features specific optimizations:

* Gradient Checkpointing: Enabled on the Generator to save activation memory.

* Frozen Encoders: The Question Encoder is frozen during fine-tuning to prioritize Generator adaptation.

* Mixed Precision (FP16): Used to reduce memory footprint and increase throughput.

# Coding Phase

### Prerequisites: Dependencies, Imports, Configs

In [1]:
%%capture
%pip install ipywidgets

# === Core ML stack (CUDA 12.4 compatible) ===
%pip install -q \
  torch==2.5.1+cu124 \
  torchvision==0.20.1+cu124 \
  torchaudio==2.5.1+cu124 \
  --index-url https://download.pytorch.org/whl/cu124

# === Hugging Face ecosystem (compatible set) ===
%pip install -q \
  transformers==4.38.2 \
  accelerate==0.27.2 \
  datasets==2.18.0 \
  sentence-transformers==2.6.1

# === Retrieval & numeric stack ===
%pip install -q \
  faiss-cpu==1.13.2 \
  numpy==1.26.4 \
  pandas==2.2.1

# === Evaluation Metrics ===
%pip install rouge_score absl-py

In [2]:
import os, traceback, json, shutil, tempfile
import re
import random
import time
import traceback
import faiss
import torch
import pandas as pd
import pyarrow as pa
import os
import traceback
import numpy as np
import shutil
import ipywidgets as widgets

from tqdm.auto import tqdm
from IPython.display import display
from torch.utils.data import DataLoader
from functools import partial
from pathlib import Path
from typing import List, Dict, Optional, Any
from datasets import Dataset, load_dataset, load_from_disk
from transformers import (
    RagTokenizer,
    RagRetriever,
    RagTokenForGeneration,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DPRContextEncoder,
    DPRContextEncoderTokenizer,
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer,
    get_linear_schedule_with_warmup,
    Adafactor
)

#### Run Config


In [3]:
# Style helper to prevent text cutoff
style = {'description_width': 'initial'}

# Define the widgets with the style applied
eda_widget = widgets.Dropdown(
    options=[True, False],
    value=False,
    description='Enable Step-by-Step EDA:',
    style=style
)

enable_persistence_embeddings_widget = widgets.Dropdown(
    options=[True, False],
    value=True,
    description='Persist/Load Embeddings & Index:',
    style=style
)

enable_persistence_finetuned_widget = widgets.Dropdown(
    options=[True, False],
    value=False,
    description='Persist/Load Finetuned Model:',
    style=style
)

device_gpu_cuda_enabled_widget = widgets.Dropdown(
    options=[True, False],
    value=True,
    description='Enable GPU (CUDA):',
    style=style
)

In [4]:
PERSIST_DIR = "persistence"
os.makedirs(PERSIST_DIR, exist_ok=True)
dataset_path = os.path.join(PERSIST_DIR, "rag_dataset")
index_path = os.path.join(PERSIST_DIR, "rag_doc_index.faiss")
id_map_path = os.path.join(PERSIST_DIR, "doc_id_map.json")
index_path_from_datasets = os.path.join(PERSIST_DIR, "rag_doc_index_from_datasets.faiss")

### Select Run Config

In [ ]:
# Display them vertically (VBox) so everything aligns nicely
print("Select your configuration:")
display(widgets.VBox([
    eda_widget, # Enable full step by step EDA with data analysis outputs
    device_gpu_cuda_enabled_widget, # Enable GPU (CUDA)
    enable_persistence_embeddings_widget,  # Enable computation of FAISS index and dataset with embeddings (if not, just load the files)
    enable_persistence_finetuned_widget, # Enable computation and training of finetuned RAG model (if not, just load the files)
]))

Select your configuration:


### Config Details

In [6]:
enable_EDA = eda_widget.value
enable_persistence_embeddings = enable_persistence_embeddings_widget.value
enable_persistence_finetuned = enable_persistence_finetuned_widget.value
enable_gpu_cuda = device_gpu_cuda_enabled_widget.value

In [7]:
if enable_gpu_cuda:
    if torch.cuda.is_available():
        device = torch.device("cuda")
        os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
        print("✅ Success: GPU (CUDA) is enabled and available.")
        print(f"   Device Name: {torch.cuda.get_device_name(0)}")
        print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    else:
        print("⚠️ Warning: You selected GPU, but Torch could not find a CUDA device.")
        print("   Falling back to CPU mode automatically.")
        device = torch.device("cpu")
else:
    os.environ["OMP_NUM_THREADS"] = "6"
    os.environ["MKL_NUM_THREADS"] = "6"
    os.environ["OPENBLAS_NUM_THREADS"] = "6"
    os.environ["NUMEXPR_NUM_THREADS"] = "6"

    torch.set_num_threads(6)
    torch.set_num_interop_threads(2)
    device = torch.device("cpu")
print("device:", device)

✅ Success: GPU (CUDA) is enabled and available.
   Device Name: NVIDIA GeForce GTX 1650 Ti
   VRAM: 4.29 GB
device: cuda


### Dataset Source
We utilize the **Goodreads Books** dataset (via `BrightData/Goodreads-Books`). This provides rich metadata including:
* `name` (Title)
* `summary` (Description)
* `kindle_price` (Price)

### Data Preparation & Insights
Before training, we perform EDA to ensure data quality:
1.  **Missing Values:** We identify rows with missing descriptions or prices. Missing descriptions are dropped as they cannot serve as retrieval contexts.
2.  **Price Normalization:** Raw price fields often contain currency symbols or formatting errors. We clean these into float values.
3.  **Distribution:** We analyze the text length of descriptions to determine optimal chunking strategies (aiming for ~100 words per passage for DPR).

## 3. Dataset and Exploratory Data Analysis (EDA)

### Dataset and Exploratory Data Analysis (EDA)
Dataset fields used: `title`, `description`, `price_numeric`.
EDA performed: dataset size, missing value counts, description length distribution, price distribution percentiles.


In [8]:
def load_raw_dataset():
    # Can replace with other dataset (need to continue full EDA)
    raw_ds = load_dataset("BrightData/Goodreads-Books", split="train[:10000]")
    raw_df = pd.DataFrame(raw_ds)

    # Create a new DataFrame with only relevant columns
    df_products = pd.DataFrame({
        "title": raw_df['name'],
        "description": raw_df['summary'],
        "price": raw_df['kindle_price']
    })
    df_products = df_products.reset_index(drop=True)
    return df_products

In [9]:
if enable_EDA:
    df_products = load_raw_dataset()
    df_products.head()

In [10]:
if enable_EDA:
    df_products.describe(include='all')

In [11]:
def clean_description(df_products):
    # Keep only rows with non-null descriptions
    df_products_clean = df_products[df_products['description'].notnull()].copy()

    # Reset the index
    df_products_clean.reset_index(drop=True, inplace=True)
    return df_products_clean

In [12]:
if enable_EDA:
    df_products_clean = clean_description(df_products)
    print("New dataset shape:", df_products_clean.shape)
    print("Missing descriptions after cleaning:", df_products_clean['description'].isnull().sum())

In [13]:
if enable_EDA:
    df_products_clean.describe(include='all')

In [14]:
if enable_EDA:
    # Top 10 titles
    print("Top 10 Titles:")
    print(df_products_clean['title'].value_counts().head(10))

    print("\nTop 10 Descriptions:")
    print(df_products_clean['description'].value_counts().head(10))

In [15]:
def replace_price(df_products_clean):
    # Replace 'None', None, and empty strings with np.nan
    df_products_clean['price'] = df_products_clean['price'].replace(['None', None, ''], np.nan)
    return df_products_clean

In [16]:
if enable_EDA:
    df_products_clean = replace_price(df_products_clean)

    missing_percentage = df_products_clean.isna().mean() * 100
    print(missing_percentage)

In [17]:
if enable_EDA:
    # Show all unique values and their frequency
    print(df_products_clean['price'].value_counts())

In [18]:
def clean_price(price):
    if price is None or str(price).strip() in ['', 'None']:
        return np.nan
    # Convert to string
    price_str = str(price).strip()
    # Remove $ sign, commas, and quotes
    price_str = price_str.replace("$", "").replace(",", "").replace('"','')
    try:
        return float(price_str)
    except:
        return np.nan

In [19]:
if enable_EDA:
    # Apply robust cleaning
    df_products_clean['price_numeric'] = df_products_clean['price'].apply(clean_price)

    # Check number of valid numeric prices
    print("Valid numeric prices:", df_products_clean['price_numeric'].notna().sum())
    print("Missing prices:", df_products_clean['price_numeric'].isna().sum())

    # Show top 20 unique values for verification
    print(df_products_clean['price_numeric'].value_counts().head(20))

In [20]:
if enable_EDA:
    # Step 4: compute percentiles only on non-null values
    p35 = df_products_clean['price_numeric'].quantile(0.35)
    p65 = df_products_clean['price_numeric'].quantile(0.65)
    print(f"35th percentile: {p35}, 65th percentile: {p65}")

    # Step 5: fill missing prices with random values between p35 and p65
    missing_mask = df_products_clean['price_numeric'].isna()
    num_missing = missing_mask.sum()
    df_products_clean.loc[missing_mask, 'price_numeric'] = np.random.uniform(p35, p65, size=num_missing)

    # Verify
    print("Number of missing prices after filling:", df_products_clean['price_numeric'].isna().sum())
    df_products_clean['price_numeric'].describe()

In [21]:
def create_rag_text_column(df_products_clean):
    # Create the RAG text column
    df_products_clean['rag_text'] = (
        "Title: " + df_products_clean['title'] +
        " Description: " + df_products_clean['description'] +
        " Price: " + (df_products_clean['price_numeric'] * 5)
            .round(2)
            .astype(str) + " lei"
    )
    return df_products_clean

In [22]:
if enable_EDA:
    df_products_clean = create_rag_text_column(df_products_clean)
    for i, text in enumerate(df_products_clean['rag_text'][:5].to_list()):
        print(f"--- Document {i+1} ---")
        print(text)
        print()

In [23]:
if enable_EDA:
    df_products_clean.info()

In [24]:
def create_rag_dataframe(df_products_clean):
    df_rag = df_products_clean[["title", "rag_text"]].rename(columns={"rag_text": "text"}).copy().reset_index(drop=True)
    return df_rag

In [25]:
if enable_EDA:
    df_rag = create_rag_dataframe(df_products_clean)
    df_rag.info()

## 4. ETL (Extract, Transform, Load)

To convert raw data into a RAG-compatible knowledge base, we implement the following pipeline:

### 1. Text Chunking
DPR works best with passages of approximately 100 words. We split long book descriptions into smaller chunks. This ensures the retriever focuses on specific plot points rather than getting lost in long texts.

### 2. Passage Formatting
We construct a structured text representation for each document:
> `Title: [Title] | Chunk: [N] | [Description Fragment] | Price: [Price]`

This format ensures the generator has access to the title and price explicitly when reading the retrieved text.

### 3. Indexing (FAISS)
* **Encoding:** We use `facebook/dpr-ctx_encoder-single-nq-base` to encode every passage into a 768-dimensional vector.
* **Indexing:** We build a **FAISS IndexFlatIP** (Inner Product) index. This allows for fast, exact similarity search during the retrieval phase.

In [26]:
def chunk_text(text: str, chunk_size_words: int = 100) -> List[str]:
    words = re.split(r'\s+', text.strip())
    chunks = []
    for i in range(0, len(words), chunk_size_words):
        chunk = " ".join(words[i:i+chunk_size_words])
        if chunk:
            chunks.append(chunk)
    return chunks

In [27]:
def get_documents_chunks(df_rag: pd.DataFrame) -> Dataset:
    docs = []
    for idx, row in df_rag.iterrows():
        title = str(row['title']).strip()
        desc = str(row['text']).strip()
        chunks = chunk_text(desc, chunk_size_words=100) or [desc]
        for i, chunk in enumerate(chunks):
            doc_id = f"{idx}_{i}"
            text = f"Title: {title} | Chunk: {i} | {chunk}"
            docs.append({"id": doc_id, "title": title, "text": text})
    return Dataset.from_list(docs)

In [28]:
if enable_EDA:
    documents = get_documents_chunks(df_rag)
    print("document chunks:", len(documents))

In [29]:
def clean_price_column(df_products_clean):
    df_products_clean['price_numeric'] = df_products_clean['price'].apply(clean_price)
    p35 = df_products_clean['price_numeric'].quantile(0.35)
    p65 = df_products_clean['price_numeric'].quantile(0.65)
    missing_mask = df_products_clean['price_numeric'].isna()
    num_missing = missing_mask.sum()
    df_products_clean.loc[missing_mask, 'price_numeric'] = np.random.uniform(p35, p65, size=num_missing)
    return df_products_clean

### Get documents from raw dataset 

In [30]:
def get_df_rag_from_raw_dataset():
    df_products = load_raw_dataset()
    df_products = clean_description(df_products)
    df_products = replace_price(df_products)
    df_products = clean_price_column(df_products)
    df_products_clean = create_rag_text_column(df_products)
    df_rag = create_rag_dataframe(df_products_clean)
    return df_rag

In [31]:
def get_documents_from_raw_dataset():
    df_rag = get_df_rag_from_raw_dataset()
    documents = get_documents_chunks(df_rag)

    print("get_documents_from_raw_dataset finished")
    return documents

In [32]:
if not enable_persistence_embeddings:
    documents = get_documents_from_raw_dataset()

### Code: DPR context encoder and FAISS index creation

In [33]:
def get_ctx_encoder_and_tokenizer():
    ctx_model_name = "facebook/dpr-ctx_encoder-single-nq-base"
    ctx_tokenizer = DPRContextEncoderTokenizer.from_pretrained(ctx_model_name)
    ctx_encoder = DPRContextEncoder.from_pretrained(ctx_model_name).to(device)
    ctx_encoder.eval()
    return ctx_encoder, ctx_tokenizer

ctx_encoder, ctx_tokenizer = get_ctx_encoder_and_tokenizer()

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'DPRQuestionEncoderTokenizer'. 
The class this function is called from is 'DPRContextEncoderTokenizer'.
Some weights of the model checkpoint at facebook/dpr-ctx_encoder-single-nq-base were not used when initializing DPRContextEncoder: ['ctx_encoder.bert_model.pooler.dense.bias', 'ctx_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRContextEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRContextEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification mod

In [34]:
def encode_documents(dataset, batch_size: int = 32, device: str = device):
    if len(dataset) == 0:
        raise ValueError("The `documents` dataset is empty. Check ETL step.")
    embs_list = []
    ids = []
    total = len(dataset)
    try:
        for start in range(0, total, batch_size):
            end = min(start + batch_size, total)
            batch = dataset[start:end]
            # Prefer column access for HF Dataset slices
            texts = batch["text"] if isinstance(batch, dict) or hasattr(batch, "__getitem__") else [d["text"] for d in batch]
            if isinstance(texts, list) and len(texts) == 0:
                continue
            inputs = ctx_tokenizer(
                texts,
                padding=True,
                truncation=True,
                return_tensors="pt",
                max_length=512
            ).to(device)
            with torch.no_grad():
                outputs = ctx_encoder(**inputs)
                # DPR context encoder returns pooled output in .pooler_output
                if hasattr(outputs, "pooler_output"):
                    batch_emb = outputs.pooler_output.cpu().numpy()
                elif hasattr(outputs, "last_hidden_state"):
                    # fallback: mean-pool last_hidden_state
                    last = outputs.last_hidden_state.cpu().numpy()
                    batch_emb = last.mean(axis=1)
                else:
                    raise RuntimeError("Unexpected DPR encoder output structure: " + str(type(outputs)))
            embs_list.append(batch_emb.astype("float32"))
            # preserve ids in same order as embeddings
            ids.extend(batch["id"])
            # lightweight progress print
            if (start // batch_size) % 10 == 0:
                print(f"Encoded {end}/{total} documents")
        embeddings = np.vstack(embs_list).astype("float32")
        # normalize for cosine similarity (IndexFlatIP)
        faiss.normalize_L2(embeddings)
        return embeddings, ids
    except RuntimeError as e:
        # Commonly OOM on GPU; try smaller batch or CPU fallback
        print("RuntimeError during encoding:", str(e))
        # traceback.print_exc() # Optional: keep if you want full logs
        
        # We can simply check for OOM and strictly fallback without touching CUDA cache
        if "out of memory" in str(e).lower():
            print("OOM detected. Retrying on CPU with smaller batch_size=8.")
            return encode_documents(dataset, batch_size=8, device=device)
        raise

In [35]:
def create_documents_embeddings():
    # Run encoding with diagnostics
    doc_embeddings, doc_ids = encode_documents(documents, batch_size=32, device=device)
    print("doc_embeddings shape:", doc_embeddings.shape)
    print("num doc_ids:", len(doc_ids))

    # Validate shapes
    if doc_embeddings.shape[0] != len(doc_ids):
        raise RuntimeError("Mismatch: number of embeddings != number of ids")

    embedding_dim = doc_embeddings.shape[1]
    print("embedding_dim:", embedding_dim)
    return doc_embeddings, doc_ids, embedding_dim

In [36]:
def create_faiss_index(doc_embeddings, doc_ids, embedding_dim):
    # Build FAISS index (Inner Product on normalized vectors = cosine similarity)
    index = faiss.IndexFlatIP(embedding_dim)
    index.add(doc_embeddings)
    
    faiss.write_index(index, index_path)
    with open(id_map_path, "w") as f:
        json.dump(doc_ids, f)

    print("FAISS index created and saved. index.ntotal =", index.ntotal)
    return index

### Create Document Embeddings and Faiss Index

In [37]:
def create_embeddings_and_faiss_index():
    doc_embeddings, doc_ids, embedding_dim = create_documents_embeddings()
    faiss_index = create_faiss_index(doc_embeddings, doc_ids, embedding_dim)
    
    # Ensure embeddings are float32 numpy arrays
    emb_array_list = [np.asarray(e, dtype=np.float32) for e in doc_embeddings]
    documents_with_emb = documents.add_column("embeddings", emb_array_list)
    print("create_embeddings_and_faiss_index finished")
    return doc_embeddings, doc_ids, embedding_dim, documents_with_emb, faiss_index

In [38]:
if not enable_persistence_embeddings:
    doc_embeddings, doc_ids, embedding_dim, documents_with_emb, faiss_index = create_embeddings_and_faiss_index()

### Save Persistence

In [39]:
def load_generator_and_dpr_query_encoder():
    # load generator and DPR query encoder
    generator_name = "facebook/bart-large"
    generator_tokenizer = AutoTokenizer.from_pretrained(generator_name)
    generator = AutoModelForSeq2SeqLM.from_pretrained(generator_name).to(device)

    rag_tokenizer = RagTokenizer.from_pretrained("facebook/rag-token-nq", generator_tokenizer=generator_tokenizer)

    q_model_name = "facebook/dpr-question_encoder-single-nq-base"
    q_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(q_model_name)
    q_encoder = DPRQuestionEncoder.from_pretrained(q_model_name).to(device)
    q_encoder.eval()
    return generator, rag_tokenizer, q_tokenizer, q_encoder

generator, rag_tokenizer, q_tokenizer, q_encoder = load_generator_and_dpr_query_encoder()

C:\Users\andre\AppData\Roaming\Python\Python312\site-packages\transformers\models\bart\configuration_bart.py:180: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizerFast'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called 

In [40]:
def save_dataset_with_faiss_index(documents_with_emb: Dataset, index_path=index_path_from_datasets):
    # Add a FAISS index on the embeddings column
    documents_with_emb.add_faiss_index(column="embeddings")

    # Persist the datasets-managed FAISS index to a .faiss file (save BEFORE dropping)
    documents_with_emb.get_index("embeddings").save(index_path)
    print("Saved index to", index_path)

    # Drop the index so the dataset can be saved
    # Do NOT assign the return value because drop_index may return None (mutates in place)
    documents_with_emb.drop_index("embeddings")

    # Now save the dataset to disk
    os.makedirs(dataset_path, exist_ok=True)
    documents_with_emb.save_to_disk(dataset_path)
    print("Saved dataset to", dataset_path)

    # Save doc id map for later reference
    doc_id_map = [d.get("id", i) for i, d in enumerate(documents_with_emb)]
    # Ensure it saves inside PERSIST_DIR
    map_path = os.path.join(PERSIST_DIR, "doc_id_map.json") 
    with open(map_path, "w") as f:
        json.dump(doc_id_map, f)
    print("Saved doc_id_map.json with", len(doc_id_map), "ids")
    return dataset_path, index_path

In [41]:
def save_persistence(
    doc_embeddings: np.ndarray,
    doc_ids: list,
    index: faiss.Index,
    dataset=None,
    out_dir: str = PERSIST_DIR,
    save_dataset: bool = True,
    dataset_num_proc: int = 1
) -> Dict[str, str]:
    """
    Persist embeddings, doc_id_map, FAISS index and optional HF Dataset.
    Uses temporary files (with proper suffixes) and atomic os.replace to avoid the
    Windows tmp-name + numpy extension bug that caused FileNotFoundError.
    Returns dict with saved paths.
    """
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    emb_npy = out / "doc_embeddings.npy"
    emb_npz = out / "doc_embeddings.npz"
    id_map = out / "doc_id_map.json"
    faiss_path = out / "rag_doc_index.faiss"
    dataset_dir = out / "rag_dataset"
    metadata_path = out / "index_metadata.json"

    # --- 1) Save embeddings atomically using NamedTemporaryFile with proper suffix ---
    # np.save will write exactly to the filename given if it already has .npy suffix.
    with tempfile.NamedTemporaryFile(delete=False, suffix=".npy", dir=str(out)) as tmpf:
        tmp_npy_path = Path(tmpf.name)
    try:
        np.save(str(tmp_npy_path), doc_embeddings)
        os.replace(str(tmp_npy_path), str(emb_npy))
    finally:
        if tmp_npy_path.exists():
            tmp_npy_path.unlink(missing_ok=True)

    # compressed .npz (use a separate tmp file with .npz suffix)
    with tempfile.NamedTemporaryFile(delete=False, suffix=".npz", dir=str(out)) as tmpf2:
        tmp_npz_path = Path(tmpf2.name)
    try:
        np.savez_compressed(str(tmp_npz_path), embeddings=doc_embeddings)
        os.replace(str(tmp_npz_path), str(emb_npz))
    finally:
        if tmp_npz_path.exists():
            tmp_npz_path.unlink(missing_ok=True)

    # --- 2) Save doc id map atomically ---
    with tempfile.NamedTemporaryFile(delete=False, suffix=".json", dir=str(out), mode="w", encoding="utf-8") as tmpid:
        tmp_id_path = Path(tmpid.name)
        json.dump(doc_ids, tmpid, ensure_ascii=False)
    try:
        os.replace(str(tmp_id_path), str(id_map))
    finally:
        if tmp_id_path.exists():
            tmp_id_path.unlink(missing_ok=True)

    # --- 3) Save FAISS index atomically ---
    with tempfile.NamedTemporaryFile(delete=False, suffix=".faiss", dir=str(out)) as tmpf3:
        tmp_faiss_path = Path(tmpf3.name)
    try:
        faiss.write_index(index, str(tmp_faiss_path))
        os.replace(str(tmp_faiss_path), str(faiss_path))
    finally:
        if tmp_faiss_path.exists():
            tmp_faiss_path.unlink(missing_ok=True)

    saved_dataset_dir = ""
    # --- 4) Save HF Dataset robustly (atomic move from temp dir). Use num_proc=1 on Windows/notebooks.
    if save_dataset and dataset is not None:
        tmpdir = tempfile.mkdtemp(prefix="rag_dataset_tmp_", dir=str(out))
        try:
            dataset.save_to_disk(tmpdir, num_proc=dataset_num_proc)
            if dataset_dir.exists():
                shutil.rmtree(dataset_dir)
            shutil.move(tmpdir, dataset_dir)
            saved_dataset_dir = str(dataset_dir)
        except Exception:
            # cleanup and fallback to parquet/jsonl
            if os.path.exists(tmpdir):
                shutil.rmtree(tmpdir)
            try:
                df = dataset.to_pandas()
                parquet_path = out / "rag_dataset_backup.parquet"
                jsonl_path = out / "rag_dataset_backup.jsonl"
                df.to_parquet(str(parquet_path), index=False)
                df.to_json(str(jsonl_path), orient="records", lines=True)
                saved_dataset_dir = str(parquet_path)
            except Exception as e:
                raise RuntimeError("Failed to persist HF Dataset (save_to_disk and parquet fallback failed).") from e

    # --- 5) metadata (atomic) ---
    meta = {
        "ntotal": int(index.ntotal),
        "dim": int(index.d),
        "emb_shape": [int(s) for s in doc_embeddings.shape],
        "num_doc_ids": len(doc_ids),
        "faiss_path": str(faiss_path),
        "dataset_dir": saved_dataset_dir,
    }
    with tempfile.NamedTemporaryFile(delete=False, suffix=".json", dir=str(out), mode="w", encoding="utf-8") as tmpmeta:
        tmp_meta_path = Path(tmpmeta.name)
        json.dump(meta, tmpmeta, indent=2)
    try:
        os.replace(str(tmp_meta_path), str(metadata_path))
    finally:
        if tmp_meta_path.exists():
            tmp_meta_path.unlink(missing_ok=True)

    return {
        "embeddings_npy": str(emb_npy),
        "embeddings_npz": str(emb_npz),
        "doc_id_map": str(id_map),
        "faiss_index": str(faiss_path),
        "dataset_dir": saved_dataset_dir,
        "metadata": str(metadata_path),
    }

In [42]:
def load_persistence_cell(out_dir: str = PERSIST_DIR, load_embeddings: bool = True, load_dataset: bool = True):
    """
    Strict loader for persisted artifacts. Performs assertions and raises on mismatch.
    Exposes these globals on success: doc_embeddings, doc_ids, index, documents_with_emb, metadata.
    Returns a dict with loaded objects and paths.
    """
    out = Path(out_dir)
    assert out.exists(), f"Persistence directory not found: {out.resolve()}"

    emb_npy = out / "doc_embeddings.npy"
    emb_npz = out / "doc_embeddings.npz"
    id_map = out / "doc_id_map.json"
    faiss_path = out / "rag_doc_index.faiss"
    dataset_dir = out / "rag_dataset"
    metadata_path = out / "index_metadata.json"

    # metadata (optional)
    metadata: Dict[str, Any] = {}
    if metadata_path.exists():
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)

    # doc ids (required)
    assert id_map.exists(), f"doc_id_map.json not found in {out.resolve()}"
    with open(id_map, "r", encoding="utf-8") as f:
        doc_ids = json.load(f)
    assert isinstance(doc_ids, list), "doc_id_map must be a JSON list"
    assert len(doc_ids) > 0, "doc_id_map is empty"

    # FAISS index (required)
    assert faiss_path.exists(), f"FAISS index file not found in {out.resolve()}"
    index = faiss.read_index(str(faiss_path))
    assert hasattr(index, "ntotal") and index.ntotal > 0, "Loaded FAISS index has invalid ntotal"
    assert hasattr(index, "d") and index.d > 0, "Loaded FAISS index has invalid dimension"

    # Optional: load embeddings
    doc_embeddings: Optional[np.ndarray] = None
    if load_embeddings:
        if emb_npy.exists():
            doc_embeddings = np.load(str(emb_npy))
        elif emb_npz.exists():
            with np.load(str(emb_npz)) as arr:
                if "embeddings" in arr:
                    doc_embeddings = arr["embeddings"]
                else:
                    keys = list(arr.keys())
                    assert keys, "Compressed embeddings archive is empty"
                    doc_embeddings = arr[keys[0]]
        else:
            doc_embeddings = None

    # Optional: load dataset
    documents_with_emb = None
    if load_dataset:
        if dataset_dir.exists():
            documents_with_emb = load_from_disk(str(dataset_dir))
        else:
            documents_with_emb = None

    # --- Strict assertions and consistency checks ---
    # 1) index <-> doc_ids
    assert index.ntotal == len(doc_ids), f"FAISS ntotal ({index.ntotal}) != len(doc_id_map) ({len(doc_ids)})"

    # 2) metadata consistency (if metadata present)
    if metadata:
        if "ntotal" in metadata:
            assert int(metadata["ntotal"]) == index.ntotal, f"metadata.ntotal ({metadata['ntotal']}) != FAISS ntotal ({index.ntotal})"
        if "dim" in metadata:
            assert int(metadata["dim"]) == index.d, f"metadata.dim ({metadata['dim']}) != FAISS dim ({index.d})"
        if "num_doc_ids" in metadata:
            assert int(metadata["num_doc_ids"]) == len(doc_ids), f"metadata.num_doc_ids ({metadata['num_doc_ids']}) != len(doc_id_map) ({len(doc_ids)})"

    # 3) embeddings vs index
    if doc_embeddings is not None:
        assert isinstance(doc_embeddings, np.ndarray), "Loaded embeddings must be a numpy array"
        assert doc_embeddings.ndim == 2, f"Embeddings must be 2D array, got shape {doc_embeddings.shape}"
        assert doc_embeddings.shape[0] == index.ntotal, f"embeddings rows ({doc_embeddings.shape[0]}) != FAISS ntotal ({index.ntotal})"
        assert doc_embeddings.shape[1] == index.d, f"embeddings dim ({doc_embeddings.shape[1]}) != FAISS dim ({index.d})"
        assert np.isfinite(doc_embeddings).all(), "Embeddings contain non-finite values"
        assert not np.isnan(doc_embeddings).any(), "Embeddings contain NaN values"

    # 4) dataset vs index
    if documents_with_emb is not None:
        assert len(documents_with_emb) == index.ntotal, f"dataset rows ({len(documents_with_emb)}) != FAISS ntotal ({index.ntotal})"
        # expected columns (at least these)
        expected_cols = {"id", "text", "embeddings"}
        present_cols = set(documents_with_emb.column_names)
        missing = expected_cols - present_cols
        assert not missing, f"Dataset missing expected columns: {missing}"

    # 5) doc_id types and uniqueness
    assert len(set(doc_ids)) == len(doc_ids), "doc_id_map contains duplicate ids"
    for i, did in enumerate(doc_ids[:50]):
        assert isinstance(did, (str, int)), f"doc_id at position {i} has unexpected type: {type(did)}"

    # 6) FAISS self-retrieval spot-check (deterministic indices)
    if doc_embeddings is not None:
        emb = np.asarray(doc_embeddings).astype("float32")
        sample_rows = [0, max(0, index.ntotal // 2), max(0, index.ntotal - 1)]
        sample_rows = [r for r in sample_rows if 0 <= r < index.ntotal]
        for r in sample_rows:
            q = emb[r:r+1]
            # If you normalized before indexing, uncomment the next line:
            # faiss.normalize_L2(q)
            D, I = index.search(q, 1)
            returned = int(I[0][0])
            assert returned == r, f"Self-retrieval mismatch for row {r}: expected {r}, got {returned} (score {float(D[0][0]):.6f})"

    # Expose to globals for convenience in notebook
    g = globals()
    g["doc_embeddings"] = doc_embeddings
    g["doc_ids"] = doc_ids
    g["index"] = index
    g["documents_with_emb"] = documents_with_emb
    g["metadata"] = metadata

    # Summary print
    print("Loaded persistence (strict) from:", out.resolve())
    print("  FAISS index:", str(faiss_path), "| ntotal:", index.ntotal, "dim:", index.d)
    print("  doc_id_map:", str(id_map), "| count:", len(doc_ids))
    if doc_embeddings is not None:
        print("  embeddings:", str(emb_npy if emb_npy.exists() else emb_npz), "| shape:", doc_embeddings.shape)
    else:
        print("  embeddings: not loaded")
    if documents_with_emb is not None:
        print("  dataset:", str(dataset_dir), "| rows:", len(documents_with_emb))
    else:
        print("  dataset: not loaded or not present")
    if metadata:
        print("  metadata keys:", list(metadata.keys()))

    return {
        "doc_embeddings": doc_embeddings,
        "doc_ids": doc_ids,
        "index": index,
        "documents_with_emb": documents_with_emb,
        "metadata": metadata,
        "paths": {
            "out_dir": str(out.resolve()),
            "emb_npy": str(emb_npy) if emb_npy.exists() else "",
            "emb_npz": str(emb_npz) if emb_npz.exists() else "",
            "doc_id_map": str(id_map),
            "faiss_index": str(faiss_path),
            "dataset_dir": str(dataset_dir) if dataset_dir.exists() else "",
            "metadata": str(metadata_path) if metadata_path.exists() else "",
        }
    }

### Save and Reload Dataset (Check Persistence)

In [43]:
if not enable_persistence_embeddings:
    dataset_path, index_path = save_dataset_with_faiss_index(documents_with_emb)
    print("Saved dataset at:", dataset_path)

    paths = save_persistence(doc_embeddings=doc_embeddings, doc_ids=doc_ids, index=faiss_index, dataset=documents_with_emb)
    print("Saved artifacts:", paths)

loaded = load_persistence_cell(load_embeddings=True, load_dataset=True)

Loaded persistence (strict) from: C:\Users\andre\Desktop\ML\Project\persistence
  FAISS index: persistence\rag_doc_index.faiss | ntotal: 17305 dim: 768
  doc_id_map: persistence\doc_id_map.json | count: 17305
  embeddings: persistence\doc_embeddings.npy | shape: (17305, 768)
  dataset: persistence\rag_dataset | rows: 17305
  metadata keys: ['ntotal', 'dim', 'emb_shape', 'num_doc_ids', 'faiss_path', 'dataset_dir']


In [44]:
def load_dataset_and_faiss_index():
    print("dataset exists:", os.path.isdir(dataset_path))
    print("index file exists:", os.path.isfile(index_path))

    ds = load_from_disk(dataset_path)
    print("dataset columns:", ds.column_names)

    # Check FAISS index dimension matches encoder output
    idx = faiss.read_index(index_path)
    print("faiss ntotal:", idx.ntotal, "dataset rows:", len(ds))

    # Only attach if counts match
    if idx.ntotal != len(ds):
        raise RuntimeError("FAISS ntotal != dataset rows. Do not attach — rebuild or fix ordering.")

    # Attach the index in memory
    ds.load_faiss_index("embeddings", index_path)
    print("Attached index. list_indexes:", ds.list_indexes())

    try:
        print("list_indexes:", ds.list_indexes())
    except Exception as e:
        print("list_indexes error:", type(e).__name__, e)

    # If index file is a faiss file, check its vector dim
    try:
        idx = faiss.read_index(index_path)
        print("faiss index d:", idx.d)
    except Exception as e:
        print("faiss read error:", type(e).__name__, e)

    # Check embeddings dtype/shape and dataset length
    print("rows:", len(ds))
    emb0 = ds["embeddings"][0]
    print("emb dtype:", np.asarray(emb0).dtype, "len:", len(emb0))

    # Ensure lengths match
    assert len(ds) == idx.ntotal, "Dataset rows != FAISS rows — ordering mismatch!"
    return ds, idx

In [45]:
def create_rag_model(dataset_path: str, index_path: str, ctx_encoder, ctx_tokenizer, generator):
    # 1. Load the dataset
    ds = load_from_disk(dataset_path)
    
    # 2. CRITICAL: Prepare Generator-compatible tokens
    # The retriever looks for 'input_ids' in the dataset to use as context.
    # We must overwrite any existing 'input_ids' (which might be DPR-based) 
    # with new ones from the BART Generator.
    gen_tokenizer = rag_tokenizer.generator
    
    def add_generator_tokens(batch):
        enc = gen_tokenizer(
            batch["text"], 
            truncation=True, 
            padding="max_length", 
            max_length=128, 
            return_tensors="np"
        )
        # Note: We name this 'input_ids' because RagRetriever specifically 
        # looks for this column to pass to the generator.
        return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}

    print("Aligning dataset context IDs with Generator vocabulary...")
    # remove_columns=['input_ids', ...] ensures we don't have conflicting columns
    cols_to_remove = [c for c in ds.column_names if c in ['input_ids', 'attention_mask', 'token_type_ids']]
    ds = ds.map(add_generator_tokens, batched=True, remove_columns=cols_to_remove)
    
    # 3. CRITICAL: Re-load the FAISS index
    print(f"Loading FAISS index from {index_path}...")
    ds.load_faiss_index("embeddings", index_path)

    # 4. Initialize Retriever
    retriever = RagRetriever.from_pretrained(
        "facebook/rag-token-nq",
        indexed_dataset=ds,
        index_name="custom",
        use_dummy_dataset=False,
        ctx_encoder=ctx_encoder,
        # CORRECTED LINE: This must be the GENERATOR's tokenizer
        ctx_encoder_tokenizer=rag_tokenizer.generator 
    )

    # 5. Create RAG Model
    rag_model = RagTokenForGeneration.from_pretrained(
        "facebook/rag-token-nq",
        retriever=retriever
    ).to(device)

    # 6. Set Generator
    generator = generator.to(device)
    rag_model.generator = generator

    print("RAG model aligned and ready.")
    return rag_model

### Code: Prepare training dataset (replace synthetic with real dialogues)

In [46]:
def make_synthetic_pairs(df, n=100):
    titles = df['title'].tolist()
    pairs = []
    for _ in range(n):
        genre = random.choice(["sci-fi","horror","romance","fantasy","mystery", "adventure", "non-fiction", "biography"])
        budget = random.choice(["under 20","under 30","under 50", "cheap", "median budget", "budget friendly", "for posh people"])
        q = f"Recommend three {genre} books for a gift, budget {budget}."
        picks = random.sample(titles, 3)
        t = " ; ".join([f"{i+1}) {p}" for i,p in enumerate(picks)]) + " ; Final: I recommend these three."
        pairs.append({"query": q, "answer": t})
    return Dataset.from_list(pairs)

df_rag = get_df_rag_from_raw_dataset()
train_ds = make_synthetic_pairs(df_rag, n=100)
print("train size:", len(train_ds))


train size: 100


### Code: Tokenization for Trainer

In [47]:
def create_tokenized_dataset(train_ds: Dataset, rag_tokenizer: RagTokenizer):
    # Use the generator tokenizer (BART) for the targets
    target_tokenizer = rag_tokenizer.generator
    
    max_source_length = 128
    max_target_length = 64
    PAD_ID = target_tokenizer.pad_token_id

    def preprocess(examples):
        # 1. Input (Question) -> RAG Question Encoder
        inputs = rag_tokenizer(
            examples["query"],
            truncation=True,
            padding="max_length",
            max_length=max_source_length,
        )

        # 2. Target (Answer) -> Generator Tokenizer
        labels = target_tokenizer(
            examples["answer"],
            truncation=True,
            padding="max_length",
            max_length=max_target_length,
        )

        # CRITICAL FIX: Do NOT convert padding to -100 for RAG-Token.
        # RAG needs valid indices (pad_token_id) for its marginalization step.
        # We leave the labels as they are (with PAD_ID).
        inputs["labels"] = labels["input_ids"]
        
        return inputs

    tokenized_train = train_ds.map(
        preprocess,
        batched=True,
        remove_columns=train_ds.column_names
    )

    tokenized_train.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )
    
    return tokenized_train, target_tokenizer

## 5. Hyperparameters

### Rationale for Low-Resource Training
Training RAG-Token (a ~500MB+ model) on a **4GB GPU** requires aggressive optimization. We deviate from standard settings (e.g., AdamW) to prioritize memory efficiency.

| Hyperparameter | Value | Justification |
| :--- | :--- | :--- |
| **Optimizer** | `Adafactor` | **Critical:** Uses ~50% less memory than AdamW (which stores momentum/variance states). Essential for 4GB VRAM. |
| **Batch Size** | `1` | Maximum possible fit on the hardware. |
| **Gradient Accumulation** | `8` | Simulates an effective batch size of 8 to stabilize gradients despite the per-device size of 1. |
| **Learning Rate** | `5e-6` | Lower than standard (typically 1e-5) to prevent catastrophic forgetting in the generator, given the small batch size. |
| **Epochs** | `5` | Sufficient for a "Micro-Run" to demonstrate convergence on the synthetic dataset. |
| **Precision** | `FP16` | Mixed Precision enabled via `torch.amp` to reduce memory usage and speed up computation. |

In [48]:
# --- Hyperparameters (Memory-Safe Mode) ---
num_epochs = 5
per_device_train_batch_size = 1
gradient_accumulation_steps = 8
learning_rate = 5e-6
weight_decay = 0.01
log_every = 5
save_every = 1000

# Output setup
output_dir = Path(PERSIST_DIR) / "rag_finetuned"
output_dir.mkdir(parents=True, exist_ok=True)

# Re-initialize model
rag_model = create_rag_model(dataset_path, index_path, ctx_encoder, ctx_tokenizer, generator)
rag_model.generator = generator

# 1. Freeze Question Encoder (Saves VRAM)
print("❄️ Freezing Question Encoder...")
for param in rag_model.question_encoder.parameters():
    param.requires_grad = False

# 2. CRITICAL: Enable Gradient Checkpointing (Saves massive memory)
print("🛡️ Enabling Gradient Checkpointing for Generator...")
rag_model.generator.gradient_checkpointing_enable()

rag_model.to(device)
rag_model.train()

Aligning dataset context IDs with Generator vocabulary...
Loading FAISS index from persistence\rag_doc_index.faiss...


C:\Users\andre\AppData\Roaming\Python\Python312\site-packages\transformers\models\bart\configuration_bart.py:180: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizerFast'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called 

RAG model aligned and ready.
❄️ Freezing Question Encoder...
🛡️ Enabling Gradient Checkpointing for Generator...


RagTokenForGeneration(
  (rag): RagModel(
    (question_encoder): DPRQuestionEncoder(
      (question_encoder): DPREncoder(
        (bert_model): BertModel(
          (embeddings): BertEmbeddings(
            (word_embeddings): Embedding(30522, 768, padding_idx=0)
            (position_embeddings): Embedding(512, 768)
            (token_type_embeddings): Embedding(2, 768)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (encoder): BertEncoder(
            (layer): ModuleList(
              (0-11): 12 x BertLayer(
                (attention): BertAttention(
                  (self): BertSelfAttention(
                    (query): Linear(in_features=768, out_features=768, bias=True)
                    (key): Linear(in_features=768, out_features=768, bias=True)
                    (value): Linear(in_features=768, out_features=768, bias=True)
                    (dropout): Dropout(p=0.

## 6. Training and Validation Loop

### Strategy
We employ a custom training loop using `PyTorch` native components rather than the Hugging Face `Trainer` to have granular control over memory management (e.g., explicit cache clearing).

### Optimization Objectives
* **Loss Function:** We minimize the negative log-likelihood of the target tokens (the recommendation response) given the query and the retrieved documents.
    $$L = - \log P(y \mid x, z)$$
* **Marginalization:** Since we use **RAG-Token**, the probability of each target token is calculated by summing the probabilities across all $K$ retrieved documents.

### Monitoring
We track the **Loss** at every step. In a RAG system, initial loss is typically high (~100-200) because the model is learning to utilize the retrieval mechanism. A downward trend indicates the model is learning to align queries with relevant documents.

In [49]:
tokenized_train, generator_tokenizer = create_tokenized_dataset(train_ds, rag_tokenizer)

# Safe collate function: ensures long dtype and label padding -> -100
pad_token_id = generator_tokenizer.pad_token_id
label_pad_token_id = -100

def collate_fn_safe(batch):
    input_ids = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
    attention_mask = [torch.tensor(b["attention_mask"], dtype=torch.long) for b in batch]
    labels = [torch.tensor(b["labels"], dtype=torch.long) for b in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=pad_token_id)

    labels = labels.clone()
    labels[labels == pad_token_id] = label_pad_token_id

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [50]:
# --- DataLoader ---
train_loader = DataLoader(tokenized_train, batch_size=per_device_train_batch_size, shuffle=True, collate_fn=collate_fn_safe, num_workers=0)

# --- Optimizer & scheduler ---
optimizer = Adafactor(
    filter(lambda p: p.requires_grad, rag_model.parameters()), 
    lr=learning_rate, 
    scale_parameter=False, 
    relative_step=False,
    weight_decay=weight_decay
)
total_steps = max(1, (len(train_loader) // gradient_accumulation_steps) * num_epochs)
warmup_percent = 10
warmup_steps = max(1, int(10 / 100 * total_steps))
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

In [51]:
# ----------------- Helpers -----------------
def ensure_pad_token(tok):
    if getattr(tok, "pad_token_id", None) is None:
        tok.pad_token = tok.eos_token

def retokenize_texts_with_generator(batch, text_col="text", max_len=128):
    enc = generator_tokenizer(batch[text_col], truncation=True, padding="max_length", max_length=max_len)
    return {"context_input_ids": enc["input_ids"], "context_attention_mask": enc["attention_mask"]}

def retokenize_answers_with_generator(batch, answer_col="answer", max_label_len=64):
    enc = generator_tokenizer(batch[answer_col], truncation=True, padding="max_length", max_length=max_label_len)
    labels = np.array(enc["input_ids"], dtype=np.int64)
    pad_id = generator_tokenizer.pad_token_id
    if pad_id is None:
        raise RuntimeError("generator_tokenizer.pad_token_id is None; set pad_token before tokenizing labels.")
    labels[labels == pad_id] = -100
    return {"labels": labels}

def has_faiss_index(ds, column="embeddings"):
    try:
        if hasattr(ds, "get_index"):
            try:
                _ = ds.get_index(column)
                return True
            except Exception:
                return False
        if hasattr(ds, "list_indexes"):
            return len(ds.list_indexes()) > 0
    except Exception:
        return False
    return False

def drop_faiss_index_if_present(ds, column="embeddings"):
    try:
        if hasattr(ds, "drop_index"):
            try:
                ds.drop_index(column)
            except Exception:
                # try dropping all indexes
                try:
                    for idx in ds.list_indexes():
                        try:
                            ds.drop_index(idx)
                        except Exception:
                            pass
                except Exception:
                    pass
    except Exception:
        pass

In [52]:
# ----------------- Preconditions -----------------
assert "rag_model" in globals(), "rag_model not found in globals"
assert "generator_tokenizer" in globals(), "generator_tokenizer not found in globals"
assert "rag_tokenizer" in globals(), "rag_tokenizer not found in globals"
assert ("documents_with_emb" in globals() or os.path.isdir("rag_dataset")), "documents_with_emb dataset not found and rag_dataset folder missing"
assert ("tokenized_train" in globals() or "train_ds" in globals() or "train_dataset" in globals()), "tokenized_train or raw train dataset not found"

# ----------------- Prepare tokenizers & info -----------------
ensure_pad_token(generator_tokenizer)
print("generator_tokenizer.pad_token_id:", generator_tokenizer.pad_token_id)
print("rag_tokenizer.pad_token_id:", getattr(rag_tokenizer, "pad_token_id", None))
vocab_size = getattr(rag_model.generator.config, "vocab_size", None)
print("generator vocab_size:", vocab_size)

# ----------------- 1) Load / retokenize contexts -----------------
ds = globals().get("documents_with_emb", None)
if ds is None and os.path.isdir("rag_dataset"):
    ds = load_from_disk("rag_dataset")
    globals()["documents_with_emb"] = ds

if ds is None:
    raise RuntimeError("No documents_with_emb dataset available to retokenize contexts.")

print("Ensure contexts tokenized for generator (context_input_ids)...")
if "context_input_ids" not in ds.column_names:
    backup_dir = Path(PERSIST_DIR) / "rag_dataset_backup_before_context_fix"
    if not backup_dir.exists() and os.path.isdir("rag_dataset"):
        try:
            shutil.copytree("rag_dataset", str(backup_dir))
            print("Backed up rag_dataset ->", backup_dir)
        except Exception:
            print("Backup failed or already exists; continuing.")
    ds = ds.map(partial(retokenize_texts_with_generator, text_col="text", max_len=128), batched=True, num_proc=1)
    globals()["documents_with_emb"] = ds
    # Save a copy without any FAISS index attached (safe)
    try:
        # ensure no index attached before saving
        if has_faiss_index(ds, "embeddings"):
            drop_faiss_index_if_present(ds, "embeddings")
        ds.save_to_disk(str(Path(PERSIST_DIR) / "rag_dataset_fixed_no_index"), num_proc=1)
        print("Saved rag_dataset_fixed_no_index (no FAISS index attached).")
    except Exception as e:
        print("Warning: could not save rag_dataset_fixed_no_index:", repr(e))
else:
    print("Dataset already contains 'context_input_ids'. Attempting to save a no-index copy.")
    try:
        if has_faiss_index(ds, "embeddings"):
            ds_copy = ds
            drop_faiss_index_if_present(ds_copy, "embeddings")
            ds_copy.save_to_disk("rag_dataset_fixed_no_index", num_proc=1)
        else:
            ds.save_to_disk("rag_dataset_fixed_no_index", num_proc=1)
        print("Saved rag_dataset_fixed_no_index.")
    except Exception as e:
        print("Warning: could not save rag_dataset_fixed_no_index:", repr(e))

# ----------------- 2) Build FAISS index in-memory (do NOT save dataset with index attached) -----------------
ds_in_memory = ds
if "embeddings" not in ds_in_memory.column_names:
    raise RuntimeError("Dataset must contain an 'embeddings' column to build a FAISS index. Add embeddings first.")

if not has_faiss_index(ds_in_memory, "embeddings"):
    print("Building FAISS index in-memory on 'embeddings' (this may take a while)...")
    try:
        ds_in_memory.add_faiss_index(column="embeddings")
        print("FAISS index built and attached in memory.")
    except Exception:
        print("Failed to build FAISS index in-memory. Traceback:")
        traceback.print_exc()
        raise RuntimeError("Could not build FAISS index in-memory. Ensure 'embeddings' column contains numeric vectors and faiss is installed.")
else:
    print("FAISS index already present in-memory; skipping build.")

# ----------------- 3) Create RagRetriever from in-memory indexed dataset -----------------
old_retriever = getattr(rag_model, "retriever", None)
ctx_enc = getattr(old_retriever, "ctx_encoder", None) if old_retriever is not None else None

print("Creating RagRetriever from in-memory indexed dataset (ctx_encoder_tokenizer=generator_tokenizer)...")
try:
    new_retriever = RagRetriever.from_pretrained(
        "facebook/rag-token-nq",
        indexed_dataset=ds_in_memory,
        index_name="custom",
        dataset_path=None,
        index_path=None,
        use_dummy_dataset=False,
        ctx_encoder=ctx_enc,
        ctx_encoder_tokenizer=generator_tokenizer
    )
    print("RagRetriever created from in-memory indexed dataset.")
except Exception:
    print("RagRetriever.from_pretrained failed. Traceback:")
    traceback.print_exc()
    raise RuntimeError("Failed to create RagRetriever from in-memory indexed dataset. See traceback above.")

# ----------------- 4) Attach retriever to rag_model (setter or rebuild) -----------------
attached = False
try:
    if hasattr(rag_model, "set_retriever"):
        rag_model.set_retriever(new_retriever)
        attached = True
        print("Attached new retriever using rag_model.set_retriever().")
except Exception:
    attached = False

if not attached:
    print("Setter not available or failed; rebuilding RagTokenForGeneration with the new retriever.")
    try:
        state = rag_model.state_dict()
        model_name_or_path = getattr(rag_model, "name_or_path", None)
        if model_name_or_path is None:
            try:
                model_name_or_path = rag_model.config.name_or_path
            except Exception:
                model_name_or_path = "facebook/rag-token-nq"
                print("Using fallback model_name_or_path:", model_name_or_path)
        new_model = RagTokenForGeneration.from_pretrained(model_name_or_path, retriever=new_retriever)
        missing_keys, unexpected_keys = new_model.load_state_dict(state, strict=False)
        print("Loaded state into new model. missing_keys:", missing_keys, "unexpected_keys:", unexpected_keys)
        new_model.to(device)
        globals()["rag_model"] = new_model
        rag_model = new_model
        print("Replaced rag_model in globals with new model instance that has the new retriever attached.")
    except Exception:
        print("Rebuild fallback failed. Traceback:")
        traceback.print_exc()
        raise RuntimeError("Failed to attach new retriever by setter or rebuild. See traceback above.")

# ----------------- 5) Retokenize labels in training dataset (if raw available) -----------------
train_raw = globals().get("train_ds") or globals().get("train_dataset") or None
if train_raw is None and "tokenized_train" in globals():
    try:
        train_raw = tokenized_train.data if hasattr(tokenized_train, "data") else None
    except Exception:
        train_raw = None

if train_raw is None:
    print("No raw train dataset found; skipping full re-tokenization of labels.")
else:
    print("Retokenizing labels in training dataset with generator_tokenizer (pad -> -100)...")
    try:
        if hasattr(train_raw, "save_to_disk"):
            train_raw.save_to_disk(str(Path(PERSIST_DIR) / "train_ds_backup_before_label_fix"))
            print("Backed up train dataset to train_ds_backup_before_label_fix")
    except Exception:
        pass

    train_fixed = train_raw.map(partial(retokenize_answers_with_generator, answer_col="answer", max_label_len=64), batched=True, num_proc=1)
    if "input_ids" not in train_fixed.column_names:
        print("train_fixed missing input_ids; re-tokenizing queries with rag_tokenizer...")
        def tok_queries(batch, qcol="query", max_len=128):
            enc = rag_tokenizer(batch[qcol], truncation=True, padding="max_length", max_length=max_len)
            return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}
        train_fixed = train_fixed.map(partial(tok_queries, qcol="query", max_len=128), batched=True, num_proc=1)

    train_fixed.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    tokenized_train = train_fixed
    globals()["tokenized_train"] = tokenized_train
    print("Replaced tokenized_train with train_fixed (labels retokenized).")


generator_tokenizer.pad_token_id: 1
rag_tokenizer.pad_token_id: None
generator vocab_size: 50265
Ensure contexts tokenized for generator (context_input_ids)...


Saving the dataset (0/1 shards):   0%|          | 0/17305 [00:00<?, ? examples/s]

Saved rag_dataset_fixed_no_index (no FAISS index attached).
Building FAISS index in-memory on 'embeddings' (this may take a while)...


  0%|          | 0/18 [00:00<?, ?it/s]

FAISS index built and attached in memory.
Creating RagRetriever from in-memory indexed dataset (ctx_encoder_tokenizer=generator_tokenizer)...


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizerFast'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'BartTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called fr

... (3 outputs left out)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

train_fixed missing input_ids; re-tokenizing queries with rag_tokenizer...


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Replaced tokenized_train with train_fixed (labels retokenized).


In [53]:
# It performs explicit retrieval -> re-tokenize contexts with generator_tokenizer -> validate -> forward.
import os, traceback, torch, numpy as np
from datasets import load_from_disk

n_docs = 5
ctx_max_len = 128
doc_text_col = "text"  # change if your dataset uses a different column for raw doc text
indexed_ds_paths = ["rag_dataset_fixed_no_index", "rag_dataset_fixed", "rag_dataset"]

def ensure_pad(tok):
    if getattr(tok, "pad_token_id", None) is None:
        tok.pad_token = tok.eos_token

def build_context_tensors_from_texts(texts, n_docs=n_docs, max_len=ctx_max_len):
    docs = texts[:n_docs] + [""] * max(0, n_docs - len(texts))
    enc = generator_tokenizer(docs, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
    return enc["input_ids"].unsqueeze(0).long().to(device), enc["attention_mask"].unsqueeze(0).long().to(device)

def try_load_indexed_dataset(indexed_ds_paths):
    for p in indexed_ds_paths:
        if os.path.isdir(p):
            try:
                return load_from_disk(p)
            except Exception:
                pass
    return globals().get("documents_with_emb", None)

def get_retriever_doc_ids(retriever, q_ids, q_att, n_docs=n_docs):
    try:
        if hasattr(retriever, "get_top_docs"):
            top = retriever.get_top_docs(q_ids, q_att, n_docs=n_docs)
            if isinstance(top, dict):
                if "doc_ids" in top:
                    return top["doc_ids"][0][:n_docs]
                if "docs" in top:
                    docs = top["docs"][0][:n_docs]
                    return [getattr(d, "id", None) or getattr(d, "doc_id", None) for d in docs]
            else:
                if hasattr(top, "doc_ids"):
                    return top.doc_ids[0][:n_docs]
                if hasattr(top, "docs"):
                    docs = top.docs[0][:n_docs]
                    return [getattr(d, "id", None) or getattr(d, "doc_id", None) for d in docs]
    except Exception:
        pass
    try:
        docs = retriever.retrieve(q_ids, q_att)
        if isinstance(docs, dict):
            if "doc_ids" in docs:
                return docs["doc_ids"][0][:n_docs]
            if "context_input_ids" in docs:
                return None
    except Exception:
        pass
    return None

def lookup_texts_by_doc_ids(indexed_ds, doc_ids):
    texts = []
    if doc_ids is None:
        return texts
    for did in doc_ids:
        if did is None:
            texts.append("")
            continue
        try:
            idx = int(did)
            texts.append(indexed_ds[idx][doc_text_col])
            continue
        except Exception:
            pass
        if "id" in indexed_ds.column_names:
            found = indexed_ds.filter(lambda x, val=did: x["id"] == val)
            if len(found) > 0:
                texts.append(found[0][doc_text_col])
                continue
        texts.append("")
    return texts

def validate_tensor_range(tensor, name, vocab_size):
    arr = tensor.cpu().numpy().ravel()
    if (arr < 0).any():
        print(f"ERROR: {name} contains negative ids (min={int(arr.min())})")
        return False
    if vocab_size is not None:
        bad = arr[arr >= vocab_size]
        if bad.size > 0:
            print(f"ERROR: {name} contains ids >= vocab_size. Sample offending ids:", np.unique(bad)[:20])
            return False
    return True

In [54]:
def ensure_pad(tok):
    if getattr(tok, "pad_token_id", None) is None:
        tok.pad_token = tok.eos_token

def build_context_tensors_from_texts(texts, n_docs=n_docs, max_len=ctx_max_len):
    docs = texts[:n_docs] + [""] * max(0, n_docs - len(texts))
    enc = generator_tokenizer(docs, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
    return enc["input_ids"].unsqueeze(0).long().to(device), enc["attention_mask"].unsqueeze(0).long().to(device)

def try_load_indexed_dataset(indexed_ds_paths):
    for p in indexed_ds_paths:
        if os.path.isdir(p):
            try:
                return load_from_disk(p)
            except Exception:
                pass
    return globals().get("documents_with_emb", None)

def get_retriever_doc_ids(retriever, q_ids, q_att, n_docs=n_docs):
    try:
        # Try various retriever access patterns
        if hasattr(retriever, "get_top_docs"):
            top = retriever.get_top_docs(q_ids, q_att, n_docs=n_docs)
            if isinstance(top, dict):
                 return top.get("doc_ids", [[]])[0][:n_docs]
    except Exception:
        pass
    try:
        docs = retriever.retrieve(q_ids, q_att)
        if isinstance(docs, dict) and "doc_ids" in docs:
            return docs["doc_ids"][0][:n_docs]
    except Exception:
        pass
    return None

def lookup_texts_by_doc_ids(indexed_ds, doc_ids):
    texts = []
    if doc_ids is None: return texts
    for did in doc_ids:
        try:
            # Assumes standard HF dataset access
            texts.append(indexed_ds[int(did)][doc_text_col])
        except:
            texts.append("")
    return texts

def validate_tensor_range(tensor, name, vocab_size):
    arr = tensor.cpu().numpy().ravel()
    if (arr < 0).any():
        print(f"ERROR: {name} contains negative ids (min={int(arr.min())})")
        return False
    if vocab_size is not None:
        bad = arr[arr >= vocab_size]
        if bad.size > 0:
            print(f"ERROR: {name} contains ids >= vocab_size. Sample: {np.unique(bad)[:5]}")
            return False
    return True

### Training helpers & checks

In [55]:
# --- HELPER 1: Prepare a Single Batch from Dataset ---
def get_sample_on_device(dataset, index, device):
    """Extracts a sample, prints diagnostics, and moves tensors to device."""
    ex = dataset[index]
    print("\n1. Diagnostics (Raw Data):")
    for k in ["input_ids", "attention_mask", "labels"]:
        if k in ex:
            arr = np.asarray(ex[k])
            print(f"   {k}: shape={arr.shape} min={int(arr.min())} max={int(arr.max())}")

    q_ids = torch.as_tensor(ex["input_ids"]).unsqueeze(0).long().to(device)
    q_att = torch.as_tensor(ex["attention_mask"]).unsqueeze(0).long().to(device)
    labs = torch.as_tensor(ex["labels"]).unsqueeze(0).long().to(device)
    
    return q_ids, q_att, labs

# --- HELPER 2: Retrieve Contexts (Vectors or Text) ---
def resolve_context(rag_model, q_ids, q_att, n_docs, indexed_ds_paths, generator_tokenizer, device):
    """Handles the complex logic of retrieving docs vs fetching text from disk."""
    print(f"\n2. Retrieving Context (n_docs={n_docs})...")
    retriever = getattr(rag_model, "retriever", None)
    if retriever is None:
        raise RuntimeError("rag_model.retriever not found.")

    # Try retrieving dense vectors directly first
    try:
        docs_out = retriever.retrieve(q_ids, q_att)
        if isinstance(docs_out, dict) and "input_ids" in docs_out:
            print("   -> Retrieved dense vectors directly.")
            ctx = np.asarray(docs_out["input_ids"])
            if ctx.ndim == 2: ctx = ctx[None, ...]
            c_ids = torch.as_tensor(ctx).long().to(device)
            
            c_att = None
            if "attention_mask" in docs_out:
                cam = np.asarray(docs_out["attention_mask"])
                if cam.ndim == 2: cam = cam[None, ...]
                c_att = torch.as_tensor(cam).long().to(device)
            return c_ids, c_att
    except Exception:
        pass

    # Fallback: Get Doc IDs and look up text on disk
    print("   -> Fallback: Looking up text from indexed dataset...")
    doc_ids = get_retriever_doc_ids(retriever, q_ids, q_att, n_docs) # Uses your existing helper
    
    # Load dataset
    indexed_ds = None
    for p in indexed_ds_paths:
        if os.path.isdir(p):
            indexed_ds = load_from_disk(p)
            break
            
    doc_texts = []
    if indexed_ds and doc_ids:
        doc_texts = lookup_texts_by_doc_ids(indexed_ds, doc_ids) # Uses your existing helper
    else:
        doc_texts = [""] * n_docs

    print(f"   -> Found text sample: {doc_texts[:1]}")
    
    # Tokenize texts
    enc = generator_tokenizer(doc_texts, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
    c_ids = enc["input_ids"].unsqueeze(0).long().to(device)
    c_att = enc["attention_mask"].unsqueeze(0).long().to(device)
    return c_ids, c_att

# --- HELPER 3: Fix Labels & Create Decoder Inputs ---
def prepare_safe_labels(labels, generator_tokenizer, model_config):
    """
    CRITICAL FIX: 
    1. Replaces -100 in labels with pad_token_id (prevents RAG-Token crash).
    2. Shifts labels right to create decoder_input_ids.
    """
    print("\n3. Preparing Safe Labels & Decoder Inputs...")
    pad_id = generator_tokenizer.pad_token_id
    dec_start_id = model_config.decoder_start_token_id
    
    # Fix 1: Safe Labels for Loss Calculation
    safe_labels = labels.clone()
    safe_labels[safe_labels == -100] = pad_id
    
    # Fix 2: Shift tokens for Decoder Input
    decoder_input_ids = labels.new_zeros(labels.shape)
    decoder_input_ids[:, 1:] = labels[:, :-1].clone()
    decoder_input_ids[:, 0] = dec_start_id
    # Ensure no -100 in input (replace with pad)
    decoder_input_ids.masked_fill_(decoder_input_ids == -100, pad_id)
    
    return safe_labels, decoder_input_ids

# --- HELPER 4: Execute Forward Pass ---
def execute_forward_pass(rag_model, inputs):
    """Runs the model and catches errors."""
    print(f"\n4. Running Forward Pass on {inputs['input_ids'].device}...")
    try:
        with torch.no_grad():
            out = rag_model(**inputs)
        print(f"✅ SUCCESS! Loss: {float(out.loss)}")
        return out
    except Exception:
        print("❌ Forward failed.")
        traceback.print_exc()
        return None

In [56]:
def run_sanity_check_clean():
    ensure_pad(generator_tokenizer)
    
    # 1. Define Paths
    paths = [
        str(Path(PERSIST_DIR) / "rag_dataset_fixed_no_index"),
        str(Path(PERSIST_DIR) / "rag_dataset_fixed"),
        str(Path(PERSIST_DIR) / "rag_dataset")
    ]

    # 2. Get Data
    q_ids, q_att, raw_labels = get_sample_on_device(tokenized_train, 0, device)

    # 3. Get Context
    c_ids, c_att = resolve_context(
        rag_model, q_ids, q_att, 
        n_docs=5, 
        indexed_ds_paths=paths, 
        generator_tokenizer=generator_tokenizer, 
        device=device
    )

    # 4. Prepare Safe Inputs (The Fixes)
    safe_labels, decoder_inputs = prepare_safe_labels(
        raw_labels, 
        generator_tokenizer, 
        rag_model.generator.config
    )

    # 5. Build Input Dict
    model_inputs = {
        "input_ids": q_ids,
        "attention_mask": q_att,
        "context_input_ids": c_ids,
        "context_attention_mask": c_att,
        "labels": safe_labels,               # <--- The fixed labels
        "decoder_input_ids": decoder_inputs, # <--- The manual inputs
        "n_docs": 5
    }

    # 6. Run
    execute_forward_pass(rag_model, model_inputs)

# Run it
run_sanity_check_clean()


1. Diagnostics (Raw Data):
   input_ids: shape=(128,) min=0 max=16755
   attention_mask: shape=(128,) min=0 max=1
   labels: shape=(64,) min=-100 max=44052

2. Retrieving Context (n_docs=5)...
   -> Fallback: Looking up text from indexed dataset...
   -> Found text sample: ['']

3. Preparing Safe Labels & Decoder Inputs...

4. Running Forward Pass on cuda:0...


C:\Users\andre\AppData\Roaming\Python\Python312\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


✅ SUCCESS! Loss: 562.923583984375


In [57]:
# ----------------- 7) Per-batch sanitizer -----------------
def sanitize_and_move_batch(batch, rag_model, generator_tokenizer):
    ensure_pad_token(generator_tokenizer)
    sanitized = {}
    
    # Use the global 'device' variable you set in the config step
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            t = v.long().to(device)
        else:
            t = torch.as_tensor(v).long().to(device)
        sanitized[k] = t
    
    # CRITICAL: RAG-Token loss crashes with -100. 
    # Ensure we strictly use pad_token_id for padding.
    if "labels" in sanitized:
        pad_id = generator_tokenizer.pad_token_id
        labels = sanitized["labels"]
        # Replace any accidental -100 with the real pad_id
        if (labels == -100).any():
            labels = labels.clone()
            labels[labels == -100] = pad_id
            sanitized["labels"] = labels
            
    return sanitized

### Actual Training

In [58]:
# ----------------- 8) Training loop (Optimized) -----------------
rag_model.to(device)
ensure_pad_token(generator_tokenizer)

print(f"Starting training on {device}...")
print(f"Targeting {num_epochs} epochs. Mixed Precision: {'Enabled' if torch.cuda.is_available() else 'Disabled'}")

# Setup Mixed Precision (Critical for 4GB VRAM)
use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda') if use_amp else None

global_step = 0
t0 = time.time()

# Calculate total steps for the progress bar
total_steps = (len(train_loader) // gradient_accumulation_steps) * num_epochs
progress_bar = tqdm(range(total_steps), desc="Training Steps", unit="step")

optimizer.zero_grad()

for epoch in range(num_epochs):
    epoch_loss = 0.0
    step_count = 0
    
    # Progress bar for the current epoch
    epoch_iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for step, batch in enumerate(epoch_iterator, start=1):
        try:
            batch = sanitize_and_move_batch(batch, rag_model, generator_tokenizer)
        except Exception as e:
            print(f"Batch error: {e}")
            continue

        # Clean inputs
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        # Forward pass with Mixed Precision (Autocast)
        with torch.amp.autocast('cuda', enabled=use_amp):
            outputs = rag_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss / gradient_accumulation_steps

        # Backward pass (Scaled)
        if use_amp:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        epoch_loss += float(loss.item()) * gradient_accumulation_steps
        step_count += 1
        
        # Update progress bar description with current loss
        epoch_iterator.set_postfix(loss=loss.item() * gradient_accumulation_steps)

        if step % gradient_accumulation_steps == 0:
            if use_amp:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(rag_model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(rag_model.parameters(), max_norm=1.0)
                optimizer.step()
            
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1
            progress_bar.update(1)

            if global_step % log_every == 0:
                elapsed = time.time() - t0
                current_loss = epoch_loss / step_count
                progress_bar.write(f"Step {global_step} | Loss: {current_loss:.4f}")

            if global_step % save_every == 0:
                ckpt = Path(output_dir) / f"checkpoint-{global_step}"
                ckpt.mkdir(parents=True, exist_ok=True)
                rag_model.save_pretrained(ckpt)
                generator_tokenizer.save_pretrained(ckpt)
                progress_bar.write(f"💾 Saved checkpoint: {ckpt}")

    # End-of-epoch checkpoint
    epoch_dir = Path(output_dir) / f"epoch-{epoch+1}"
    epoch_dir.mkdir(parents=True, exist_ok=True)
    rag_model.save_pretrained(epoch_dir)
    generator_tokenizer.save_pretrained(epoch_dir)
    
    avg_epoch_loss = epoch_loss / step_count
    progress_bar.write(f"✅ Finished Epoch {epoch+1} | Avg Loss: {avg_epoch_loss:.4f}")

print("Training complete.")

Starting training on cuda...
Targeting 5 epochs. Mixed Precision: Enabled


Training Steps:   0%|          | 0/60 [00:00<?, ?step/s]

Epoch 1/5:   0%|          | 0/100 [00:00<?, ?it/s]

C:\Users\andre\AppData\Local\Temp\ipykernel_20160\2179445009.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
C:\Users\andre\AppData\Local\Temp\ipykernel_20160\2179445009.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_mask = [torch.tensor(b["attention_mask"], dtype=torch.long) for b in batch]
C:\Users\andre\AppData\Local\Temp\ipykernel_20160\2179445009.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(b["labels"], dtype=

Step 5 | Loss: 560.8463
Step 10 | Loss: 550.6733


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 50, 'min_length': 1, 'num_beams': 4, 'bad_words_ids': [[0, 0]], 'forced_eos_token_id': 2}
Removed shared tensor {'rag.generator.model.encoder.embed_tokens.weight', 'generator.model.encoder.embed_tokens.weight', 'generator.lm_head.weight', 'rag.generator.lm_head.weight', 'generator.model.decoder.embed_tokens.weight', 'rag.generator.model.decoder.embed_tokens.weight'} while saving. This should be OK, but check by verifying that you don't receive any warning while reloading


... (17 outputs left out)

C:\Users\andre\AppData\Local\Temp\ipykernel_20160\2179445009.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(b["input_ids"], dtype=torch.long) for b in batch]
C:\Users\andre\AppData\Local\Temp\ipykernel_20160\2179445009.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_mask = [torch.tensor(b["attention_mask"], dtype=torch.long) for b in batch]
C:\Users\andre\AppData\Local\Temp\ipykernel_20160\2179445009.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(b["labels"], dtype=

Step 50 | Loss: 200.0359
Step 55 | Loss: 211.3809
Step 60 | Loss: 220.2261


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 50, 'min_length': 1, 'num_beams': 4, 'bad_words_ids': [[0, 0]], 'forced_eos_token_id': 2}


✅ Finished Epoch 5 | Avg Loss: 221.3488
Training complete.


In [59]:
# --- Manual Save of Final Model ---
final_output_dir = Path(PERSIST_DIR) / "rag_finetuned" / "final_model"
final_output_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving final model to {final_output_dir}...")

# 1. Save the RAG model (Retriever + Generator weights)
rag_model.save_pretrained(final_output_dir)

# 2. Save the Tokenizer (Critical for inference later)
generator_tokenizer.save_pretrained(final_output_dir)

print("Final model saved successfully.")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 50, 'min_length': 1, 'num_beams': 4, 'bad_words_ids': [[0, 0]], 'forced_eos_token_id': 2}


Saving final model to persistence\rag_finetuned\final_model...
Final model saved successfully.


In [60]:
def load_rag_finetuned_model():
    model_path = Path(PERSIST_DIR) / "rag_finetuned" / "final_model"
    loaded_model = RagTokenForGeneration.from_pretrained(str(model_path))
    tokenizer = RagTokenizer.from_pretrained(str(model_path))
    return loaded_model, tokenizer

## 7. Inference

### Procedure
Inference in RAG involves a multi-step pipeline:
1.  **Retrieval:** The user's query is encoded by the Question Encoder.
2.  **Search:** The top-$K$ (e.g., 5) most similar documents are retrieved from the FAISS index.
3.  **Generation:** The Generator conditions on these documents to produce a response.

### Solving "Model Drift"
During our experiments, we observed that fine-tuning the Question Encoder on a small synthetic dataset caused "Catastrophic Forgetting," where it lost the ability to understand general concepts (e.g., retrieving Erotica when asked for Sci-Fi).

**Correction Strategy:**
For inference, we discard the fine-tuned Question Encoder and reload the **base DPR encoder** (`facebook/dpr-question_encoder-single-nq-base`). This restores robust retrieval capabilities while allowing us to keep the fine-tuned Generator (BART) which has learned the conversational format.

In [61]:
def build_query(history: List[Dict[str, str]], user_utterance: str, max_turns: int = 30) -> str:
    """
    Build a single RAG query string from the last max_turns of conversation history.
    Automatically truncates to prevent overflow.
    """
    turns = history[-max_turns:]
    ctx = ""
    for t in turns:
        ctx += f"User: {t['user']} System: {t['system']} "
    ctx += f"User: {user_utterance}"
    
    # FIX 1: Safe max length check (RagTokenizer wrapper doesn't have model_max_length)
    max_len = 128
    if hasattr(rag_tokenizer, "question_encoder") and hasattr(rag_tokenizer.question_encoder, "model_max_length"):
        limit = rag_tokenizer.question_encoder.model_max_length
        if limit < 10000:
            max_len = limit

    # truncate if too long for tokenizer
    tokenized_ctx = rag_tokenizer(ctx, return_tensors="pt")
    if tokenized_ctx["input_ids"].shape[1] > max_len:
        while tokenized_ctx["input_ids"].shape[1] > max_len and len(turns) > 1:
            turns = turns[1:]  # remove oldest
            ctx = ""
            for t in turns:
                ctx += f"User: {t['user']} System: {t['system']} "
            ctx += f"User: {user_utterance}"
            tokenized_ctx = rag_tokenizer(ctx, return_tensors="pt")
    return ctx

def rag_inference(history: List[Dict[str, str]], user_utterance: str, num_beams: int = 5, max_new_tokens: int = 100, max_turns: int = 30, print_docs: bool = True) -> str:
    """
    Run a RAG query with multi-turn context.
    Returns the generated answer as a string.
    
    Args:
        print_docs: If True, prints the top retrieved documents (evidence) before the answer.
    """
    query = build_query(history, user_utterance, max_turns)
    
    # tokenize and move to device
    inputs = rag_tokenizer([query], return_tensors="pt", padding=True, truncation=True).to(device)
    
    # FIX 2: Explicit Retrieval Step (To show you the docs)
    if print_docs:
        print(f"Query context: '{query}'")
        print(f"\n--- Top Retrieved Documents (Evidence) ---")
        with torch.no_grad():
            question_hidden_states = rag_model.question_encoder(inputs["input_ids"])[0]
            docs_dict = rag_model.retriever(
                inputs["input_ids"], 
                question_hidden_states.cpu().detach().numpy(), 
                n_docs=3, 
                return_tensors="pt"
            )
            doc_ids = docs_dict["doc_ids"][0].tolist()
            try:
                # CRITICAL FIX: Convert list to numpy array.
                # The index expects an object with a .shape attribute.
                import numpy as np
                retrieved_doc_dicts = rag_model.retriever.index.get_doc_dicts(np.array(doc_ids))
                
                for i, doc in enumerate(retrieved_doc_dicts):
                    # Prints the text you wanted to see (Title, Description, Price)
                    # We check for 'text' or 'title' keys just to be safe
                    text_content = doc.get('text', doc.get('title', ''))
                    print(f"[Doc {i+1}]: {text_content}...") 
            except Exception as e:
                print(f"Could not print doc text: {e}")
                print(f"Doc IDs found: {doc_ids}")
        print("-" * 50)

    # FIX 3: Use max_new_tokens to avoid UserWarning
    outputs = rag_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        num_beams=num_beams,
        max_new_tokens=max_new_tokens, 
        early_stopping=True,
        num_return_sequences=1
    )
    
    # decode output
    return rag_tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

In [62]:
history = [
    {"user":"I need a gift for my brother","system":"What genres does he like?"},
    {"user":"He likes sci-fi and horror","system":"What's your budget?"}
]
user_utterance = "Under 30, prefer recent books."

generated = rag_inference(history, user_utterance)
print("generated:", generated)

Query context: 'User: I need a gift for my brother System: What genres does he like? User: He likes sci-fi and horror System: What's your budget? User: Under 30, prefer recent books.'

--- Top Retrieved Documents (Evidence) ---
[Doc 1]: Title: A Science Fiction Argosy | Chunk: 1 | Bob Shaw storyMan in His Time '65 Brian W. Aldiss storyManna '49 Peter Phillips storyMore Than Human '53 Theodore Sturgeon novelOne Ordinary Day, with Peanuts '55 Shirley Jackson storyRump-Titty-Titty-Tum-TAH-Tee '58 Fritz Leiber storySea Wrack '64 Edward Jesby storySomerset Dreams '69 Kate Wilhelm storyThe Cure '46 Henry Kuttner & C.L. Moore as by Lewis Padget storyThe Demolished Man '52 Alfred Bester novelThe Game of Rat & Dragon '55 Cordwainer Smith storyThe Red Queen's Race '49 Isaac Asimov storyThe Third Level '51 Jack Finney storyBernie the Faust '63 Wm Tenn story Price: 21.56 lei...
[Doc 2]: Title: Life After School Explained | Chunk: 1 | a lot of feedback from graduating seniors and recent grads. It's

C:\Users\andre\AppData\Roaming\Python\Python312\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
C:\Users\andre\AppData\Roaming\Python\Python312\site-packages\transformers\generation\utils.py:2957: UserWarning: `max_length` is deprecated in this function, use `stopping_criteria=StoppingCriteriaList([MaxLengthCriteria(max_length=max_length)])` instead.
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


generated:  sci - fi


### Code: Inspect top-K retrieved passages for provenance

In [63]:
# 1. Ensure we have the list of IDs and Passages from the dataset 'ds'
# (ds["id"] is a standard column in your RAG dataset)
doc_ids = ds["id"]
passages = ds["text"]

# 2. Ensure Index is ready
# We extract the raw FAISS index from the dataset wrapper
index = ds.get_index("embeddings").faiss_index

def inspect_top_k(query: str, k: int = 5, device: str = device, index=index, doc_ids=doc_ids, passages=passages):
    """
    Retrieve and display the top-K passages for a query.
    """
    # Tokenize
    q_inputs = q_tokenizer([query], return_tensors="pt", padding=True, truncation=True).to(device)

    # Embed
    with torch.no_grad():
        q_emb = q_encoder(**q_inputs).pooler_output.cpu().detach().numpy().astype("float32")

    # Normalize
    faiss.normalize_L2(q_emb)

    # Search
    D, I = index.search(q_emb, k)

    results = []
    for rank, idx in enumerate(I[0]):
        # Safety check for index bounds
        if idx < len(doc_ids):
            doc_id = doc_ids[idx]
            score = D[0][rank]
            snippet = passages[idx][:400]
            
            print(f"Rank {rank+1} | ID={doc_id} | score={score:.4f}")
            print(snippet)
            print("----")
            results.append((doc_id, passages[idx], score))
        else:
            print(f"Rank {rank+1} | Index {idx} out of bounds")

    return results

In [64]:
query = "Sci-fi books under 30 for a brother"
top_passages = inspect_top_k(query, k=5)

Rank 1 | ID=6332_2 | score=0.7124
Title: Passion & Danger | Chunk: 2 | Jumper's Promise by Kim HornsbySoul Seducer by Alicia DeanLove on the Ropes by Pat WhiteSomeday Somebody by Lori LegerThe Good Daughter by Diana LayneAn Eye for Danger by Christine M. Fairchild If you only have one dollar to spend on books this summer, this is it! Price: 29.02 lei
----
Rank 2 | ID=5717_1 | score=0.7454
Title: EROTICA TRICKY LADIES BUNDLE: OVER 600 PAGES | Chunk: 1 | designer who watches a sex scene on a balcony across from hers and then has the man show her secrets of her own. This collection totals over 600 pages. The word count is over 231,000 words. HUSBAND The self acclaimed world's greatest detective is back, and Abigail Dupree and Linda Harrison are with him again. Wife's and lovers and gi
----
Rank 3 | ID=1802_1 | score=0.7546
Title: Out! | Chunk: 1 | threshold. Patrick Owen is an out-and-proud charity worker with strong principles—and a newly discovered weakness for an older man. One snag: M

### Code: Evaluation helpers (generation metrics and grounding)

In [ ]:
from datasets import load_metric
bleu = load_metric("bleu")
rouge = load_metric("rouge")

def compute_generation_metrics(preds: List[str], refs: List[str]):
    # BLEU expects tokenized lists
    tokenized_preds = [p.split() for p in preds]
    tokenized_refs = [[r.split()] for r in refs]
    bleu_score = bleu.compute(predictions=tokenized_preds, references=tokenized_refs)
    rouge_score = rouge.compute(predictions=preds, references=refs)
    return {"bleu": bleu_score, "rouge": rouge_score}

print("Evaluation helpers ready")


C:\Users\andre\AppData\Local\Temp\ipykernel_20160\1360756293.py:2: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  bleu = load_metric("bleu")
C:\Users\andre\AppData\Roaming\Python\Python312\site-packages\datasets\load.py:756: FutureWarning: The repository for bleu contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.18.0/metrics/bleu/bleu.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(
C:\Users\andre\AppData\Roaming\Python\Python312\site-packages\datasets\load.py:756: FutureWarning: The repository for rouge contains custom code which must be execut

Evaluation helpers ready


In [66]:
# Test the metrics on a dummy example
sample_prediction = ["Dune is a great sci-fi book."]
sample_reference = ["Dune is an excellent science fiction novel."]

results = compute_generation_metrics(sample_prediction, sample_reference)
print("BLEU:", results["bleu"]["bleu"])
print("ROUGE-L:", results["rouge"]["rougeL"].mid.fmeasure)

BLEU: 0.0
ROUGE-L: 0.2857142857142857


In [67]:
print("Loading standard DPR encoders (fixing the drift)...")

# 1. Load the BASE encoders (Pre-trained, ignoring your fine-tuned weights)
# This restores the ability to match "Sci-Fi" queries to "Sci-Fi" documents.
q_encoder_base = DPRQuestionEncoder.from_pretrained("facebook/dpr-question_encoder-single-nq-base").to(device)
q_tokenizer_base = DPRQuestionEncoderTokenizer.from_pretrained("facebook/dpr-question_encoder-single-nq-base")

# 2. Define a clean retrieval function using the BASE encoder + EXISTING Index
def distinct_retrieval(query, n_docs=3):
    # Encode query using the CLEAN encoder
    inputs = q_tokenizer_base(query, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        q_emb = q_encoder_base(**inputs).pooler_output.cpu().numpy().astype("float32")
    
    # Normalize for cosine similarity (FAISS IndexFlatIP expects this)
    import faiss
    faiss.normalize_L2(q_emb)
    
    # Search in your EXISTING Faiss index (it's still good!)
    # We use the 'index' variable you created earlier from ds.get_index("embeddings").faiss_index
    D, I = index.search(q_emb, n_docs)
    
    # Print results
    print(f"Query: '{query}'")
    results = []
    
    # Get the dataset with embeddings (documents_with_emb or ds)
    # Fallback to 'ds' if 'documents_with_emb' isn't there
    dataset_ref = globals().get("documents_with_emb", globals().get("ds"))
    
    for rank, idx in enumerate(I[0]):
        if idx < len(dataset_ref):
            doc = dataset_ref[int(idx)]
            # Handle different column names if necessary
            text = doc.get("text", doc.get("title", ""))
            
            print(f"[Rank {rank+1}] (Score: {D[0][rank]:.4f})")
            print(f"   {text}...") # Print first 200 chars
            results.append(text)
        else:
             print(f"[Rank {rank+1}] Index {idx} out of bounds")
    return results

# 3. Test it immediately
print("\nTesting fixed retrieval...")
# This should now return ACTUAL Sci-Fi books, not "Social Theory"
docs = distinct_retrieval("Sci-fi books under 30 lei")

Loading standard DPR encoders (fixing the drift)...


Some weights of the model checkpoint at facebook/dpr-question_encoder-single-nq-base were not used when initializing DPRQuestionEncoder: ['question_encoder.bert_model.pooler.dense.bias', 'question_encoder.bert_model.pooler.dense.weight']
- This IS expected if you are initializing DPRQuestionEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DPRQuestionEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



Testing fixed retrieval...
Query: 'Sci-fi books under 30 lei'
[Rank 1] (Score: 0.7366)
   Title: Fantasy: The 100 Best Books | Chunk: 0 | Title: Fantasy: The 100 Best Books Description: A chronological list of the best fantasy provides a brief history of the genre and a description of each work's background. Fantasy books by authors from classic literature includes works of Johnathan Swift, Edgar Allan Poe, Charles Dickens, Emily Bronte, Robert Louis Stevenson, Oscar Wilde, and Franz Kafka. Price: 22.55 lei...
[Rank 2] (Score: 0.7517)
   Title: FAO Schwarz Toys For A Lifetime: Enhancing Childhood Through Play | Chunk: 2 | astronaut Charles Conrad. Price: 29.51 lei...
[Rank 3] (Score: 0.7533)
   Title: Born to Believe: God, Science, and the Origin of Ordinary and Extraordinary Beliefs | Chunk: 2 | 79.95 lei...


## 8. Conclusion and Future Directions

### Summary
We successfully implemented a **Retrieval-Augmented Generation (RAG)** pipeline on a memory-constrained environment (GTX 1650 Ti, 4GB VRAM). By leveraging **Gradient Checkpointing**, **Adafactor Optimization**, and **Mixed Precision Training**, we fine-tuned a BART generator to provide grounded recommendations based on a custom product catalog.

### Future Improvements
To transition this prototype into a production-grade recommender, we propose the following enhancements:

* **Real-World Conversational Data:** Replace synthetic (query, answer) pairs with real user interaction logs to capture natural dialogue nuances and implicit preferences.
* **Optimized Retrieval Index:** Migrate from a flat FAISS index (`IndexFlatIP`) to an approximate nearest neighbor index (e.g., **HNSW** or **IVF**) to scale to millions of items with sub-millisecond latency.
* **Cross-Encoder Reranking:** Implement a second-stage reranker (Cross-Encoder) to re-score the top-50 retrieved documents for higher precision before passing them to the generator.
* **Hyperparameter Tuning:** Systematically tune the retrieval `top_k`, learning rates, and batch sizes on a held-out development set to maximize generation metrics (BLEU/ROUGE).